# Hindustani Raga Classifier Training

**Run on**: Google Colab T4 (recommended) or local RTX 3050

## Pipeline
1. Mount Google Drive / load local features
2. Architecture A — CNN on mel spectrograms (128×T)
3. Architecture B — MLP on pitch histograms (156-dim)
4. Train with weighted cross-entropy + augmentation
5. Evaluate: accuracy, macro-F1, confusion matrix
6. Export to ONNX for local real-time inference


In [ ]:
# ─── Cell 1: Environment check & installs ────────────────────────────────────
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'mirdata', 'dtaidistance', 'seaborn'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (total):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ─── Cell 2: Mount Google Drive (Colab only) ─────────────────────────────────
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ── Copy features from Drive ──────────────────────────────────────────────
    # EDIT THIS PATH to wherever you uploaded the features/ folder
    DRIVE_FEATURES = '/content/drive/MyDrive/raga-analyzer/features'
    LOCAL_FEATURES = '/content/features'

    if not os.path.exists(LOCAL_FEATURES):
        print('Copying features from Drive...')
        import shutil
        shutil.copytree(DRIVE_FEATURES, LOCAL_FEATURES)
        print('Done.')
    else:
        print('Features already present locally.')
else:
    # Local run — adjust path if needed
    LOCAL_FEATURES = './features'

print('Features dir:', LOCAL_FEATURES)
for f in os.listdir(LOCAL_FEATURES):
    path = os.path.join(LOCAL_FEATURES, f)
    size = os.path.getsize(path) // 1024
    print(f'  {f:35s} {size:>8} KB')

In [ ]:
# ─── Cell 3: Data Loading & Inspection ───────────────────────────────────────
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load metadata
with open(os.path.join(LOCAL_FEATURES, 'metadata.json')) as f:
    META = json.load(f)

with open(os.path.join(LOCAL_FEATURES, 'label_encoder.pkl'), 'rb') as f:
    LE = pickle.load(f)

N_RAGAS = META['n_ragas']
RAGA_NAMES = META['raga_names']
print(f'Number of ragas: {N_RAGAS}')
print(f'Ragas: {RAGA_NAMES}')

# Load splits
splits = {}
for split in ('train', 'val', 'test'):
    path = os.path.join(LOCAL_FEATURES, f'{split}.npz')
    if os.path.exists(path):
        data = np.load(path)
        splits[split] = {
            'hist':  data['hist'],   # (N, 12)
            'trans': data['trans'],  # (N, 144)
            'mel':   data['mel'],    # (N, 128, T)
            'label': data['label'],  # (N,)
        }
        N = len(data['label'])
        print(f'  {split:5s}: {N:>5} samples  mel_shape={data["mel"].shape}')

CLASS_WEIGHTS = np.load(os.path.join(LOCAL_FEATURES, 'class_weights.npy'))
print(f'Class weights: min={CLASS_WEIGHTS.min():.2f}  max={CLASS_WEIGHTS.max():.2f}')

# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, split in zip(axes, ('train', 'val', 'test')):
    if split not in splits: continue
    labels = splits[split]['label']
    counts = pd.Series(labels).value_counts().sort_index()
    counts.index = [RAGA_NAMES[i] for i in counts.index]
    counts.plot(kind='barh', ax=ax, color='#f5a623', alpha=0.8)
    ax.set_title(f'{split} split ({len(labels)} clips)')
    ax.set_xlabel('Clips')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 4: Dataset classes ──────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

class RagaMelDataset(Dataset):
    """Mel spectrogram dataset for Architecture A (CNN)."""
    def __init__(self, mel, labels, augment=False):
        self.mel    = torch.from_numpy(mel).unsqueeze(1).float()  # (N,1,128,T)
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.mel[idx]
        y = self.labels[idx]

        if self.augment:
            # SpecAugment: random time & freq masking
            if torch.rand(1) > 0.5:
                freq_mask = torch.randint(0, 20, (1,)).item()
                f0 = torch.randint(0, 128 - freq_mask, (1,)).item()
                x[:, f0:f0+freq_mask, :] = 0
            if torch.rand(1) > 0.5:
                time_mask = torch.randint(0, 30, (1,)).item()
                T = x.shape[-1]
                t0 = torch.randint(0, max(1, T - time_mask), (1,)).item()
                x[:, :, t0:t0+time_mask] = 0
        return x, y


class RagaHistDataset(Dataset):
    """Pitch histogram + transition dataset for Architecture B (MLP)."""
    def __init__(self, hist, trans, labels):
        feats       = np.concatenate([hist, trans], axis=1)  # (N, 12+144)
        self.feats  = torch.from_numpy(feats).float()
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.feats[idx], self.labels[idx]

In [ ]:
# ─── Cell 5: Architecture A — Lightweight CNN ─────────────────────────────────
# Input: (B, 1, 128, T) mel spectrogram
# ~2.5M parameters — fits in 4 GB VRAM with batch=32

class RagaCNN(nn.Module):
    def __init__(self, n_classes, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),          nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),  nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 128),         nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ─── Architecture B — MLP on pitch features ───────────────────────────────────
class RagaMLP(nn.Module):
    def __init__(self, n_classes, in_dim=156):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 256),    nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


# Instantiate
model_cnn = RagaCNN(N_RAGAS).to(DEVICE)
model_mlp = RagaMLP(N_RAGAS).to(DEVICE)

n_params_cnn = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
n_params_mlp = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f'CNN params: {n_params_cnn:,}')
print(f'MLP params: {n_params_mlp:,}')

In [ ]:
# ─── Cell 6: Training setup ───────────────────────────────────────────────────
ARCH      = 'CNN'    # 'CNN' or 'MLP'
EPOCHS    = 60
BATCH     = 32
LR        = 1e-3
PATIENCE  = 12       # early stopping

# Datasets
if ARCH == 'CNN':
    train_ds = RagaMelDataset(splits['train']['mel'], splits['train']['label'], augment=True)
    val_ds   = RagaMelDataset(splits['val']['mel'],   splits['val']['label'])
    test_ds  = RagaMelDataset(splits['test']['mel'],  splits['test']['label'])
    model    = model_cnn
else:  # MLP
    train_ds = RagaHistDataset(splits['train']['hist'], splits['train']['trans'], splits['train']['label'])
    val_ds   = RagaHistDataset(splits['val']['hist'],   splits['val']['trans'],   splits['val']['label'])
    test_ds  = RagaHistDataset(splits['test']['hist'],  splits['test']['trans'],  splits['test']['label'])
    model    = model_mlp

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# Loss with class weights
cw       = torch.from_numpy(CLASS_WEIGHTS).float().to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw)

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Architecture : {ARCH}')
print(f'Train batches: {len(train_dl)}')
print(f'Val   batches: {len(val_dl)}')

In [ ]:
# ─── Cell 7: Training loop ────────────────────────────────────────────────────
from tqdm.notebook import tqdm as tqdm_nb

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_loss = float('inf')
patience_cnt  = 0
best_state    = None

for epoch in range(1, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_correct, t_total = 0., 0, 0

    for xb, yb in tqdm_nb(train_dl, desc=f'Epoch {epoch:03d} [train]', leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        t_loss    += loss.item() * len(yb)
        preds      = logits.argmax(1)
        t_correct += (preds == yb).sum().item()
        t_total   += len(yb)

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0., 0, 0

    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss   = criterion(logits, yb)
            v_loss    += loss.item() * len(yb)
            preds      = logits.argmax(1)
            v_correct += (preds == yb).sum().item()
            v_total   += len(yb)

    scheduler.step()

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    t_l   = t_loss    / t_total
    v_l   = v_loss    / v_total

    history['train_loss'].append(t_l)
    history['val_loss'].append(v_l)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    # Early stopping
    if v_l < best_val_loss:
        best_val_loss = v_l
        patience_cnt  = 0
        import copy; best_state = copy.deepcopy(model.state_dict())
        print(f'Epoch {epoch:03d}  t_loss={t_l:.4f}  v_loss={v_l:.4f}  t_acc={t_acc:.3f}  v_acc={v_acc:.3f}  ★ best')
    else:
        patience_cnt += 1
        if epoch % 5 == 0:
            print(f'Epoch {epoch:03d}  t_loss={t_l:.4f}  v_loss={v_l:.4f}  t_acc={t_acc:.3f}  v_acc={v_acc:.3f}  (patience {patience_cnt}/{PATIENCE})')
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# ─── Cell 8: Training curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_loss'], label='train', color='#f5a623')
axes[0].plot(history['val_loss'],   label='val',   color='#60a5fa')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(history['train_acc'], label='train', color='#f5a623')
axes[1].plot(history['val_acc'],   label='val',   color='#60a5fa')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 9: Evaluation — Accuracy, F1, Confusion Matrix ─────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

acc    = accuracy_score(all_labels, all_preds)
f1mac  = f1_score(all_labels, all_preds, average='macro',  zero_division=0)
f1wt   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f'\n=== Test Set Results ===')
print(f'Accuracy      : {acc:.4f} ({acc*100:.1f}%)')
print(f'Macro F1      : {f1mac:.4f}')
print(f'Weighted F1   : {f1wt:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=RAGA_NAMES, zero_division=0))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(max(8, N_RAGAS//2), max(7, N_RAGAS//2)))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=RAGA_NAMES, yticklabels=RAGA_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {ARCH}  (acc={acc*100:.1f}%)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 10: ONNX Export ────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

model.eval()
model.cpu()

ONNX_PATH = f'raga_classifier_{ARCH.lower()}.onnx'

if ARCH == 'CNN':
    # T = mel time frames — use the value from metadata
    T_target = META.get('mel_T', {}).get('train', 313)
    dummy_input = torch.zeros(1, 1, 128, T_target)
    input_names  = ['input']
    dynamic_axes = {'input': {0: 'batch', 3: 'time'}, 'output': {0: 'batch'}}
else:  # MLP
    dummy_input = torch.zeros(1, 156)
    input_names  = ['input']
    dynamic_axes = {'input': {0: 'batch'}, 'output': {0: 'batch'}}

torch.onnx.export(
    model, dummy_input, ONNX_PATH,
    input_names=input_names,
    output_names=['output'],
    dynamic_axes=dynamic_axes,
    opset_version=17,
    do_constant_folding=True,
)

# Verify
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print(f'✓ ONNX model verified: {ONNX_PATH}')

# Quick inference check
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
out  = sess.run(None, {'input': dummy_input.numpy()})[0]
print(f'✓ ONNX inference check: output shape = {out.shape}')
print(f'  Top prediction: {RAGA_NAMES[out[0].argmax()]} (logit={out[0].max():.3f})')

In [ ]:
# ─── Cell 11: Save model + metadata to Drive ──────────────────────────────────
import shutil

# Save PyTorch checkpoint too
PT_PATH = f'raga_classifier_{ARCH.lower()}.pt'
torch.save({
    'arch':        ARCH,
    'state_dict':  model.state_dict(),
    'n_ragas':     N_RAGAS,
    'raga_names':  RAGA_NAMES,
    'best_val_loss': best_val_loss,
    'history':     history,
}, PT_PATH)
print(f'✓ Saved PyTorch checkpoint: {PT_PATH}')

if IN_COLAB:
    DRIVE_OUT = '/content/drive/MyDrive/raga-analyzer/models'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for fname in [ONNX_PATH, PT_PATH, 'training_curves.png', 'confusion_matrix.png']:
        if os.path.exists(fname):
            shutil.copy(fname, DRIVE_OUT)
            print(f'  → Copied {fname} → {DRIVE_OUT}')
    print('\n✓ Files saved to Google Drive.')
    print('\nNext step:')
    print('  Download raga_classifier_cnn.onnx and place it in:')
    print('  raga-analyzer/backend/models/raga_classifier.onnx')
else:
    out_dir = os.path.join('backend', 'models')
    os.makedirs(out_dir, exist_ok=True)
    dest = os.path.join(out_dir, 'raga_classifier.onnx')
    shutil.copy(ONNX_PATH, dest)
    print(f'✓ Copied ONNX model → {dest}')

In [ ]:
# ─── Cell 12: (Optional) Per-raga accuracy report ─────────────────────────────
print('Per-raga Test Accuracy\n' + '─'*40)
for i, name in enumerate(RAGA_NAMES):
    mask  = np.array(all_labels) == i
    if mask.sum() == 0:
        continue
    raga_preds = np.array(all_preds)[mask]
    raga_true  = np.array(all_labels)[mask]
    acc_r = (raga_preds == raga_true).mean()
    n_r   = mask.sum()
    bar   = '█' * int(acc_r * 20) + '░' * (20 - int(acc_r * 20))
    print(f'  {name:<25} {bar}  {acc_r*100:.0f}%  (n={n_r})')